# GPU Diagnostics in C++

**First:** Runtime → Change runtime type → **T4 GPU**. Then run the cells in order.

## 1. Check the GPU is there

In [ ]:
!nvidia-smi

## 2. Write the C++ / CUDA program

In [ ]:
%%writefile diag.cu
// =====================================================
//  Simple GPU Diagnostic Tool (C++ / CUDA)
//
//  How to run:  ./diag
//  To test faults: set injectFault to true below,
//  then compile and run again.
// =====================================================

#include <iostream>
#include <cuda_runtime.h>
#include <nvml.h>

using namespace std;


// =====================================================
//  PART 1: Settings shared by the whole program
// =====================================================

bool injectFault = false;      // change to true to test faults
nvmlDevice_t gpu;              // ID card for the GPU being tested

int testsRun = 0;
int testsFailed = 0;


// =====================================================
//  PART 2: Health checks (ask the GPU about itself)
// =====================================================

bool testTemperature() {
    unsigned int temp = 0;
    nvmlReturn_t result = nvmlDeviceGetTemperature(gpu, NVML_TEMPERATURE_GPU, &temp);

    // If the reading itself failed, temp would stay 0 and wrongly PASS.
    // So: no reading means no pass.
    if (result != NVML_SUCCESS) {
        cout << "   ERROR: could not read temperature" << endl;
        return false;
    }

    unsigned int limit = 85;
    if (injectFault == true) {
        limit = 10;            // fault: no GPU is this cold
    }

    cout << "   temperature: " << temp << " C (limit " << limit << " C)" << endl;

    if (temp >= limit) {
        cout << "   ERROR: GPU is too hot" << endl;
        return false;
    }
    return true;
}

bool testFan() {
    unsigned int speed = 0;
    nvmlDeviceGetFanSpeed(gpu, &speed);   // stays 0 if the GPU has no fan sensor

    if (injectFault == true) {
        speed = 150;           // fault: impossible speed
    }

    cout << "   fan speed: " << speed << "%" << endl;

    if (speed > 100) {
        cout << "   ERROR: fan speed is impossible" << endl;
        return false;
    }
    return true;
}


// =====================================================
//  PART 3: GPU kernels (small functions that run ON the GPU)
//  Thousands of copies run at once, one per number.
// =====================================================

__global__ void fillMemory(unsigned int* data, int n, unsigned int pattern) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;   // which number am I?
    if (i < n) {
        data[i] = pattern;
    }
}

__global__ void addNumbers(float* a, float* b, float* result, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;   // which number am I?
    if (i < n) {
        result[i] = a[i] + b[i];
    }
}


// =====================================================
//  PART 4: GPU tests (make the GPU do work, check the answers)
// =====================================================

// Write patterns into GPU memory, read them back, count mistakes
bool testMemory() {
    int n = 1000000;
    unsigned int* data;
    cudaMallocManaged(&data, n * sizeof(unsigned int));   // memory both CPU and GPU can use

    unsigned int patterns[4] = {0x00000000, 0xFFFFFFFF, 0xAAAAAAAA, 0x55555555};
    int mistakes = 0;

    for (int p = 0; p < 4; p++) {
        fillMemory<<<(n + 255) / 256, 256>>>(data, n, patterns[p]);
        cudaDeviceSynchronize();          // wait for the GPU to finish

        if (injectFault == true) {
            data[1000] = 12345;           // fault: corrupt one value
        }

        for (int i = 0; i < n; i++) {
            if (data[i] != patterns[p]) {
                mistakes++;
            }
        }
    }

    cudaFree(data);
    cout << "   memory: " << mistakes << " wrong values" << endl;

    if (mistakes > 0) {
        cout << "   ERROR: GPU memory is not storing values correctly" << endl;
        return false;
    }
    return true;
}

// Add numbers on the GPU, then check every answer on the CPU
bool testCompute() {
    int n = 1000000;
    float *a, *b, *result;
    cudaMallocManaged(&a, n * sizeof(float));
    cudaMallocManaged(&b, n * sizeof(float));
    cudaMallocManaged(&result, n * sizeof(float));

    for (int i = 0; i < n; i++) {
        a[i] = i * 1.0f;
        b[i] = i * 2.0f;
    }

    addNumbers<<<(n + 255) / 256, 256>>>(a, b, result, n);
    cudaDeviceSynchronize();              // wait for the GPU to finish

    if (injectFault == true) {
        result[500] = -1.0f;              // fault: one wrong answer
    }

    int wrong = 0;
    for (int i = 0; i < n; i++) {
        if (result[i] != a[i] + b[i]) {
            wrong++;
        }
    }

    cudaFree(a);
    cudaFree(b);
    cudaFree(result);
    cout << "   compute: " << wrong << " wrong answers" << endl;

    if (wrong > 0) {
        cout << "   ERROR: GPU gave wrong answers" << endl;
        return false;
    }
    return true;
}


// =====================================================
//  PART 5: Print PASS or FAIL for one test
// =====================================================

void report(const char* name, bool passed) {
    testsRun++;
    if (passed == true) {
        cout << "   [" << name << "] PASS" << endl;
    } else {
        cout << "   [" << name << "] FAIL" << endl;
        testsFailed++;
    }
}


// =====================================================
//  PART 6: main - count the GPUs, test each one
// =====================================================

int main() {

    nvmlInit();                                  // start NVIDIA's library

    unsigned int count = 0;
    nvmlDeviceGetCount(&count);                  // 1. count the GPUs
    cout << "Found " << count << " GPU(s)" << endl << endl;

    for (unsigned int i = 0; i < count; i++) {   // 2. visit each one
        nvmlDeviceGetHandleByIndex(i, &gpu);     // 3. get its ID card
        cudaSetDevice(i);                        //    and point CUDA at it too
        cout << "Checking GPU " << i << endl;

        report("temp", testTemperature());       // 4. run the checks
        report("fan", testFan());
        report("memory", testMemory());
        report("compute", testCompute());
        cout << endl;
    }

    cout << "Summary: " << testsRun << " run, " << testsFailed << " failed" << endl;

    nvmlShutdown();
    return 0;
}

## 3. Compile it

In [ ]:
!nvcc -o diag diag.cu -L/usr/local/cuda/lib64/stubs -lnvidia-ml

## 4. Run the tests

In [ ]:
!./diag

## 5. Testing faults
In the program cell (step 2), change `bool injectFault = false;` to `true`.
Then rerun steps 2, 3 and 4. The temperature, fan, memory and compute tests should now FAIL on purpose.
Change it back to `false` when you're done.

## 6. Connect to your website
1. In Vercel, set up your site and its `DEPLOY_KEY` (see the website's README).
2. In Colab, click the **key icon** in the left sidebar, add a secret named `DEPLOY_KEY` with the same value, and switch on **Notebook access**.
3. Paste your website address below and run the cell.

In [ ]:
# Your Vercel website address (no slash at the end)
WEBSITE = "https://your-project.vercel.app"

# The deploy key is read from Colab's secrets panel, so it never appears in your code
from google.colab import userdata
DEPLOY_KEY = userdata.get("DEPLOY_KEY")

print("Connected settings for", WEBSITE)

## 7. Control panel
- **Run diagnostics** tests the GPU here in Colab.
- **Deploy to website** runs the normal and fault runs and sends the results to your website.
- **Start listening for runs** lets the website's **Run again** button work. Keep this notebook open while listening.

In [ ]:
!pip install -q gradio

In [ ]:
import subprocess
import threading
import time
from datetime import datetime, timezone

import gradio as gr
import requests

tool_lock = threading.Lock()   # makes sure only one run happens at a time
listening = False              # true while Colab is listening to the website


# ================= Running the C++ tool =================

def run_tool(inject):
    # Set injectFault in the C++ program, then compile and run it
    code = open("diag.cu").read()
    if inject == True:
        code = code.replace("bool injectFault = false;", "bool injectFault = true;")
    else:
        code = code.replace("bool injectFault = true;", "bool injectFault = false;")
    open("diag.cu", "w").write(code)

    subprocess.run(["nvcc", "-o", "diag", "diag.cu",
                    "-L/usr/local/cuda/lib64/stubs", "-lnvidia-ml"], check=True)
    return subprocess.run(["./diag"], capture_output=True, text=True).stdout


def read_results(output):
    # Turn the printed lines into a list of GPUs and their tests
    gpus = []
    details = []
    for line in output.splitlines():
        line = line.strip()
        if line.startswith("Checking GPU"):
            gpus.append({"id": line.replace("Checking ", ""), "tests": []})
            details = []
        elif line.startswith("[") and (line.endswith("PASS") or line.endswith("FAIL")):
            name = line[1:line.index("]")]
            result = line.split()[-1]
            gpus[-1]["tests"].append({"name": name, "result": result, "details": details})
            details = []
        elif line != "" and len(gpus) > 0:
            details.append(line)

    passed = 0
    failed = 0
    for gpu in gpus:
        for test in gpu["tests"]:
            if test["result"] == "PASS":
                passed = passed + 1
            else:
                failed = failed + 1
    return {"gpus": gpus, "passed": passed, "failed": failed, "output": output}


def collect_results():
    # Run the fault run first and the normal run last, so the program ends in normal mode
    with tool_lock:
        fault = read_results(run_tool(True))
        healthy = read_results(run_tool(False))

    info = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True).stdout.strip().splitlines()[0].split(", ")

    return {
        "sample": False,
        "generated": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
        "gpu": {"name": info[0], "driver": info[1], "memory": info[2]},
        "github": "",      # paste your GitHub repo link here
        "runs": {"healthy": healthy, "fault": fault},
    }


# ================= Talking to the website =================

def send_to_website(results):
    r = requests.post(WEBSITE + "/api/results", json=results,
                      headers={"x-deploy-key": DEPLOY_KEY}, timeout=30)
    if r.status_code == 401:
        return "❌ The website rejected the deploy key. Check DEPLOY_KEY matches in Colab and Vercel."
    if r.status_code != 200:
        return "❌ The website returned an error: " + r.text
    return "✅ Sent to " + WEBSITE + " at " + results["generated"]


def listen_loop():
    # Check the website every 5 seconds; run the tests when someone clicked Run again
    while listening == True:
        try:
            r = requests.post(WEBSITE + "/api/poll",
                              headers={"x-deploy-key": DEPLOY_KEY}, timeout=15)
            if r.status_code == 200 and r.json()["run"] == True:
                print("Website asked for a run. Running...")
                print(send_to_website(collect_results()))
            elif r.status_code != 200:
                print("Website said:", r.status_code, r.text)
        except Exception as e:
            print("Couldn't reach the website:", e)
        time.sleep(5)


# ================= Button actions =================

def run_here(inject):
    with tool_lock:
        output = run_tool(inject)
        if inject == True:
            run_tool(False)          # put the program back to normal mode
    run = read_results(output)

    rows = []
    for gpu in run["gpus"]:
        for test in gpu["tests"]:
            if test["result"] == "PASS":
                rows.append([gpu["id"], test["name"], "✅ PASS"])
            else:
                rows.append([gpu["id"], test["name"], "❌ FAIL"])

    total = run["passed"] + run["failed"]
    if run["failed"] == 0:
        summary = "## ✅ All " + str(total) + " tests passed"
    else:
        summary = "## ❌ " + str(run["failed"]) + " of " + str(total) + " tests failed"
    return summary, rows, output


def deploy_clicked():
    return send_to_website(collect_results())


def start_listening():
    global listening
    if listening == False:
        listening = True
        threading.Thread(target=listen_loop, daemon=True).start()
    return "🟢 Listening. Click Run again on your website and Colab will run the tests."


def stop_listening():
    global listening
    listening = False
    return "⚪ Not listening. The website's Run again button will wait until you start listening."


# ================= The control panel =================

with gr.Blocks(title="GPU Diagnostics") as app:
    gr.Markdown("# GPU Diagnostics control panel")

    gr.Markdown("### Try it here")
    inject = gr.Checkbox(label="Inject faults (tests should fail on purpose)")
    run_button = gr.Button("Run diagnostics", variant="primary")
    summary = gr.Markdown()
    table = gr.Dataframe(headers=["GPU", "Test", "Result"])
    log = gr.Textbox(label="Full output", lines=12)

    gr.Markdown("### Your website")
    deploy_button = gr.Button("Deploy to website")
    deploy_message = gr.Markdown()
    with gr.Row():
        listen_button = gr.Button("Start listening for runs")
        stop_button = gr.Button("Stop listening")
    listen_message = gr.Markdown("⚪ Not listening.")

    run_button.click(run_here, inputs=inject, outputs=[summary, table, log])
    deploy_button.click(deploy_clicked, outputs=deploy_message)
    listen_button.click(start_listening, outputs=listen_message)
    stop_button.click(stop_listening, outputs=listen_message)

app.launch()